In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model_dir = "models"
dataset_dir = "dataset"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=model_dir,
    torch_dtype="auto",
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
    lora_alpha=32,
    lora_dropout=0.05
)

model = get_peft_model(model, lora_config)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=model_dir
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main", cache_dir=dataset_dir)

train_dataset = dataset["train"].shuffle(seed=10).select(range(200))

In [ ]:
def prepare_sample(sample, tokenizer):
    messages = [
        {"role": "system", "content": "You are a math assistant. Solve the problem step by step. At the end, write your final numeric answer on a new line starting with ####. Example: #### num"},
        {"role": "user", "content": sample["question"]}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    
    prompt_tokens = tokenizer(prompt, padding=False, truncation=False)
    
    return {
        "input_ids": prompt_tokens["input_ids"],
        "attention_mask": prompt_tokens["attention_mask"],
        "prompt": prompt,
        "answer": sample["answer"]
    }

In [ ]:
updated_train_dataset = train_dataset.map(
    prepare_sample,
    remove_columns=train_dataset.column_names,
    fn_kwargs={"tokenizer": tokenizer})

In [ ]:
max_len=512

filtered_train_dataset = updated_train_dataset.filter(lambda x: 
    len(x["input_ids"]) < max_len
)

In [ ]:
from torch.nn.utils.rnn import pad_sequence
import torch

def single_padding(batch):
    token_ids = [torch.tensor(item['input_ids']) for item in batch]
    attention_masks = [torch.tensor(item['attention_mask']) for item in batch]
    prompts = [item['prompt'] for item in batch]
    answers = [item['answer'] for item in batch]
    
    token_ids = pad_sequence(
        token_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
        padding_side='left'
    )
    
    attention_masks = pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0,
        padding_side='left'
    )
    
    return {
        "token_ids": token_ids,
        "attention_masks":attention_masks,
        "prompts":prompts,
        "answers":answers
    }

In [ ]:
from torch.utils.data import DataLoader

batch_size = 4

train_dataloader = DataLoader(
    filtered_train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=single_padding
)

In [ ]:
from GRPO import GRPOTrain
import re

def extract_answer(text):
    match = re.search(r'####\s*(\-?\d+\.?\d*)', text)
    if match:
        return match.group(1).strip()
    numbers = re.findall(r'\-?\d+\.?\d*', text)
    return numbers[-1] if numbers else None

def get_rewards(sequences, answers, G):
    decoded = tokenizer.batch_decode(sequences, skip_special_tokens=True)
    repeated_answers = [a for a in answers for _ in range(G)]
    rewards = []
    for text, answer in zip(decoded, repeated_answers):
        predicted = extract_answer(text)
        expected = extract_answer(str(answer))
        rewards.append(1.0 if predicted == expected else 0.0)
    rewards = torch.tensor(rewards, device=sequences.device).float()
    return rewards.reshape(-1, G)

epochs = 10
save_path = "/checkpoints"

GRPOTrain(
    model,
    train_dataloader,
    get_rewards,
    save_path,
    epochs
)